In [ ]:
# VIIRS -> vegetation-filtered rolling hosted feature layer
# Schedule this notebook to run once per hour in ArcGIS Notebook Server.
# First run: creates a new empty hosted feature layer.
# Every run: adds only unseen recent detections and removes detections older than 7 days.

import json
import math
from datetime import datetime, timezone
import requests

import pandas as pd
import requests
from arcgis.features import FeatureLayer, FeatureLayerCollection, GeoAccessor, GeoSeriesAccessor
from arcgis.geometry import MultiPoint
from arcgis.raster import ImageryLayer
from arcgis.gis import GIS

# ----------------------------- SETTINGS -----------------------------
SERVICE_NAME = "viirs_vegetation_hotspots_notebook"
SERVICE_TITLE = "VIIRS Vegetation Hotspots (Notebook)"
SERVICE_TAG = "viirs-hourly-notebook-layer"

ROLLING_HOURS = 168
SOURCE_OVERLAP_HOURS = 1
VEGETATION_CODES = {10, 20, 30, 40, 90, 95, 100}
VEGETATION_THRESHOLD = 0.5
EDIT_BATCH_SIZE = 200
WORLDCOVER_SAMPLE_BATCH = 900  # getSamples has an approximate 1000-location limit

VIIRS_URL = (
    "https://services9.arcgis.com/RHVPKKiFTONKtxq3/arcgis/rest/services/"
    "Satellite_VIIRS_Thermal_Hotspots_and_Fire_Activity/FeatureServer/0/query"
)

COUNTRIES_URL = (
    "https://demoportal12.esri.de/server/rest/services/Hosted/"
    "ne_50m_admin_0_countries/FeatureServer/0"
)
COUNTRY_FIELD = "SOVEREIGNT"

# ArcGIS Living Atlas: European Space Agency WorldCover 2021 Land Cover
WORLDCOVER_URL = ("https://tiledimageservices.arcgis.com/P3ePLMYs2RVChkJx/arcgis/rest/services/"
                      "European_Space_Agency_WorldCover_2021_Land_Cover_WGS84_7/ImageServer"
)

WORLDCOVER_CLASS = {
    10: "Tree cover",
    20: "Shrubland",
    30: "Grassland",
    40: "Cropland",
    50: "Built-up",
    60: "Bare or sparse vegetation",
    70: "Snow and ice",
    80: "Permanent water bodies",
    90: "Herbaceous wetland",
    95: "Mangroves",
    100: "Moss and lichen",
}

# Connections used throughout the run.
gis = GIS("home")
agol = GIS("https://www.arcgis.com")
countries = FeatureLayer(COUNTRIES_URL, gis)
worldcover = ImageryLayer(WORLDCOVER_URL)

now = datetime.now(timezone.utc)
now_ms = int(now.timestamp() * 1000)
http = requests.Session()
http.headers.update({"User-Agent": "VIIRS-ArcGIS-Notebook/1.0"})



In [ ]:
# ------------------------ HOSTED FEATURE LAYER -----------------------
def get_or_create_layer(gis):
    """Use the existing VIIRS layer or create it if it does not exist."""
    # Search for the layer created by this notebook
    items = gis.content.search(
        query=f'tags:"{SERVICE_TAG}" AND owner:{gis.users.me.username}',
        item_type="Feature Layer",
        max_items=1
    )

    # Use existing layer
    if items:
        print(f"Using existing layer: {items[0].title}")
        return items[0], items[0].layers[0]

    # No layer found -> create it below
    print("No existing layer found. Creating a new one...")

    item = gis.content.create_service(
        name=SERVICE_NAME,
        service_type="featureService",
        service_description="Rolling vegetation-filtered VIIRS hotspots.",
        has_static_data=False,
        max_record_count=4000,
        capabilities="Query,Create,Update,Delete,Editing",
        wkid=4326,
    )
    item.update(
        item_properties={
            "title": SERVICE_TITLE,
            "snippet": "Rolling 7-day VIIRS hotspots filtered with ESA WorldCover 2021.",
            "tags": [SERVICE_TAG, "VIIRS", "wildfire", "WorldCover"],
        }
    )

    layer_definition = {
        "name": "VIIRS vegetation hotspots",
        "type": "Feature Layer",
        "geometryType": "esriGeometryPoint",
        "objectIdField": "OBJECTID",
        "fields": [
            {"name": "OBJECTID", "alias": "OBJECTID", "type": "esriFieldTypeOID", "nullable": False, "editable": False},
            {"name": "detection_id", "alias": "Detection ID", "type": "esriFieldTypeString", "length": 100, "nullable": False},
            {"name": "acq_time", "alias": "Acquisition time (UTC)", "type": "esriFieldTypeDate", "nullable": False},
            {"name": "hours_old", "alias": "Hours old", "type": "esriFieldTypeDouble", "nullable": True},
            {"name": "frp", "alias": "Fire Radiative Power", "type": "esriFieldTypeDouble", "nullable": True},
            {"name": "landcover_center_class", "alias": "Land-cover center class", "type": "esriFieldTypeString", "length": 64, "nullable": True},
            {"name": "country", "alias": "Country", "type": "esriFieldTypeString", "length": 128, "nullable": True},
        ],
        "extent": {
            "xmin": -180,
            "ymin": -90,
            "xmax": 180,
            "ymax": 90,
            "spatialReference": {"wkid": 4326},
        },
        "timeInfo": {
            "startTimeField": "acq_time",
            "timeReference": {"timeZone": "UTC", "respectsDaylightSaving": False},
            "timeInterval": 1,
            "timeIntervalUnits": "esriTimeUnitsHours",
        },
    }

    flc = FeatureLayerCollection.fromitem(item)
    result = flc.manager.add_to_definition({"layers": [layer_definition]})
    if not result.get("success"):
        raise RuntimeError(f"Could not create layer: {result}")

    item = gis.content.get(item.id)
    layer = item.layers[0]
    result = layer.manager.add_to_definition(
        {
            "indexes": [
                {"name": "ux_detection_id", "fields": "detection_id", "isAscending": True, "isUnique": True},
                {"name": "ix_acq_time", "fields": "acq_time", "isAscending": True, "isUnique": False},
            ]
        }
    )
    if not result.get("success"):
        raise RuntimeError(f"Could not create indexes: {result}")

    print(f"Created new empty hosted feature layer: {item.title} ({item.id})")
    return item, layer

In [ ]:
def viirs_request(params):
    # Send request directly to the VIIRS FeatureServer
    response = requests.get(
        VIIRS_URL,
        params={**params, "f": "json"},
        timeout=(30, 300)
    )
    response.raise_for_status()

    result = response.json()

    if "error" in result:
        raise RuntimeError(result["error"].get("message", str(result["error"])))

    return result


def newest_source_age():
    statistics = json.dumps([
        {
            "statisticType": "min",
            "onStatisticField": "hours_old",
            "outStatisticFieldName": "min_age"
        }
    ])

    result = viirs_request({
        "where": "1=1",
        "outStatistics": statistics,
        "returnGeometry": "false"
    })

    return int(result["features"][0]["attributes"]["min_age"])


def detection_id(satellite, acq_time, longitude, latitude):
    # Stable ID independent of ArcGIS OBJECTID
    return f"{satellite}|{float(acq_time):.0f}|{float(longitude):.5f}|{float(latitude):.5f}"


def download_new_viirs(max_age):
    """Download newest VIIRS detections plus the overlap."""
    fields = "OBJECTID,acq_time,satellite,confidence,frp,scan,track,hours_old"
    where = f"hours_old <= {max_age} AND confidence IN ('nominal','high')"

    rows = []
    offset = 0

    while True:
        result = viirs_request({
            "where": where,
            "outFields": fields,
            "returnGeometry": "true",
            "outSR": 4326,
            "orderByFields": "OBJECTID ASC",
            "resultOffset": offset,
            "resultRecordCount": 16000
        })

        page = result.get("features", [])

        for feature in page:
            a = feature["attributes"]
            g = feature["geometry"]

            lon = float(g["x"])
            lat = float(g["y"])
            acq = float(a["acq_time"])
            sat = str(a.get("satellite") or "")

            rows.append({
                "detection_id": detection_id(sat, acq, lon, lat),
                "longitude": lon,
                "latitude": lat,
                "acq_time_ms": acq,
                "frp": np.nan if a.get("frp") is None else float(a["frp"]),
            })

        if not result.get("exceededTransferLimit", False):
            break

        offset += len(page)

    return (
        pd.DataFrame(rows)
        .drop_duplicates("detection_id")
        .reset_index(drop=True)
        if rows else pd.DataFrame()
    )


def remove_existing(df, layer, lookback_hours):
    """Remove detections already stored in the hosted layer."""
    if df.empty:
        return df

    # Extra hour avoids boundary problems caused by integer hours_old values
    existing = layer.query(
        where=f"acq_time >= CURRENT_TIMESTAMP - INTERVAL '{lookback_hours + 1}' HOUR",
        out_fields="detection_id",
        return_geometry=False,
        return_all_records=True
    )

    known = {
        feature.attributes["detection_id"]
        for feature in existing.features
        if feature.attributes.get("detection_id")
    }

    return df[~df["detection_id"].isin(known)].reset_index(drop=True)

In [ ]:
# -------------------------- WORLDCOVER -------------------------------
import arcpy


def filter_worldcover(df):
    """Keep VIIRS points whose center pixel is a vegetation class."""
    if df.empty:
        return df

    df = df.reset_index(drop=True)

    # Temporary point feature class containing one point per VIIRS detection
    points = r"memory\viirs_points"
    samples = r"memory\worldcover_samples"

    arcpy.management.Delete(points)
    arcpy.management.Delete(samples)

    arcpy.management.CreateFeatureclass(
        "memory",
        "viirs_points",
        "POINT",
        spatial_reference=4326
    )
    arcpy.management.AddField(points, "row_id", "LONG")

    # Add the VIIRS center coordinates
    with arcpy.da.InsertCursor(points, ["SHAPE@XY", "row_id"]) as cursor:
        for i, r in df.iterrows():
            cursor.insertRow(((r.longitude, r.latitude), i))

    # Get one WorldCover pixel value for each VIIRS point
    arcpy.sa.Sample(
        WORLDCOVER_URL,
        points,
        samples,
        "NEAREST",
        "row_id"
    )

    # Find the WorldCover value field created by Sample
    fields = [
        f.name for f in arcpy.ListFields(samples)
        if f.name not in {"OBJECTID", "row_id"}
    ]
    value_field = fields[-1]

    # Read sampled WorldCover classes
    codes = {}
    with arcpy.da.SearchCursor(samples, ["row_id", value_field]) as cursor:
        for row_id, code in cursor:
            codes[row_id] = code

    df["landcover_code"] = df.index.map(codes)
    df["landcover_center_class"] = (
        df["landcover_code"]
        .map(WORLDCOVER_CLASS)
        .fillna("Unknown")
    )

    # Keep only vegetation classes
    keep = df["landcover_code"].isin(VEGETATION_CODES)

    print(f"WorldCover retained {keep.sum()}/{len(df)} new detections")

    return df[keep].drop(columns="landcover_code").reset_index(drop=True)

In [ ]:
# ----------------------------- COUNTRY -------------------------------
def add_country(df):
    """Assign Natural Earth SOVEREIGNT to each hotspot."""
    if df.empty:
        return df

    # Download country polygons from the hosted feature layer
    country_df = countries.query(
        where="1=1",
        out_fields="SOVEREIGNT",
        return_geometry=True
    ).sdf

    # Convert VIIRS coordinates to a Spatially Enabled DataFrame
    points = pd.DataFrame.spatial.from_xy(
        df.copy(),
        x_column="longitude",
        y_column="latitude",
        sr=4326
    )

    # Point-in-polygon join
    joined = points.spatial.join(
        country_df[["SOVEREIGNT", "SHAPE"]],
        how="left",
        op="intersects"
    )

    df = df.copy()
    df["country"] = joined["SOVEREIGNT"].fillna("").values

    return df

In [ ]:
# ---------------------------- SAVE DATA ------------------------------
def add_new_features(df, layer):
    """Add new VIIRS detections to the hosted layer."""
    if df.empty:
        return 0

    features = [
        {
            "attributes": {
                "detection_id": r.detection_id,
                "acq_time": int(r.acq_time_ms),
                "hours_old": round((NOW_MS - r.acq_time_ms) / 3_600_000, 1),
                "frp": None if pd.isna(r.frp) else float(r.frp),
                "landcover_center_class": r.landcover_center_class,
                "country": r.country,
            },
            "geometry": {
                "x": float(r.longitude),
                "y": float(r.latitude),
                "spatialReference": {"wkid": 4326},
            },
        }
        for _, r in df.iterrows()
    ]

    for start in range(0, len(features), EDIT_BATCH_SIZE):
        result = layer.edit_features(
            adds=features[start:start + EDIT_BATCH_SIZE],
            rollback_on_failure=True
        )

        if not all(x["success"] for x in result["addResults"]):
            raise RuntimeError(result)

    return len(features)


def maintain_rolling_layer(layer):
    """Delete detections older than 7 days and update their age."""

    # Delete everything outside the rolling 7-day window
    layer.delete_features(
        where=f"acq_time < CURRENT_TIMESTAMP - INTERVAL '{ROLLING_HOURS}' HOUR",
        return_delete_results=False
    )

    # Recalculate age of the remaining detections
    layer.calculate(
        where="1=1",
        calc_expression={
            "field": "hours_old",
            "sqlExpression": "ROUND((CURRENT_TIMESTAMP - acq_time) * 24, 1)"
        }
    )

In [ ]:
# ------------------------------ RUN ---------------------------------
gis = GIS("home")
item, layer = get_or_create_layer(gis)

# Keep the hosted layer itself as the rolling persistent state.
maintain_rolling_layer(layer)
count_before = layer.query(where="1=1", return_count_only=True)

# Only query the newest source block + one overlap hour.
min_age = newest_source_age()
max_age = min_age + SOURCE_OVERLAP_HOURS
new = download_new_viirs(max_age)
print(f"Downloaded {len(new)} source detections (hours_old <= {max_age})")

# Do expensive spatial work only for detections that are not already stored.
new = remove_existing(new, layer, lookback_hours=max(max_age + 3, 6))
print(f"Unseen detections: {len(new)}")
new = filter_worldcover(new)
new = add_country(new)# ------------------------------ RUN ---------------------------------
gis = GIS("home")
item, layer = get_or_create_layer(gis)

# Remove features older than 7 days
maintain_rolling_layer(layer)

# Download newest VIIRS detections + overlap hour
min_age = newest_source_age()
max_age = min_age + SOURCE_OVERLAP_HOURS

new = download_new_viirs(max_age)
print(f"Downloaded: {len(new)}")

# Remove detections already stored in the hosted layer
new = remove_existing(new, layer, lookback_hours=max_age)
print(f"New detections: {len(new)}")

# Apply spatial filters only to genuinely new detections
new = filter_worldcover(new)
new = add_country(new)

# Add remaining detections to the hosted layer
added = add_new_features(new, layer)

# Server-side count; ArcGIS does not download all features for this
total = layer.query(
    where="1=1",
    return_count_only=True
)

print(f"Added: {added}")
print(f"Total features: {total}")
added = add_new_features(new, layer)

# Refresh the rolling window once more after the update.
maintain_rolling_layer(layer)
count_after = layer.query(where="1=1", return_count_only=True)

print(
    json.dumps(
        {
            "run_time_utc": NOW.strftime("%Y-%m-%dT%H:%M:%SZ"),
            "feature_layer_item_id": item.id,
            "features_before": count_before,
            "features_added": added,
            "features_after": count_after,
        },
        indent=2,
    )
)